In [3]:
import numpy as np

# ==============================================================================
# MANUAL DE OPERACIONES CORPORATIVAS (NOTAS DE CLASE INTEGRADAS)
# ==============================================================================
#
# CONCEPTOS CLAVE TRADUCIDOS A NEGOCIO:
# 1. CLASS (El Plano): Es el molde. No es la oficina real, son las instrucciones
#    de cómo construirla (ej. "Toda oficina debe tener escritorio y silla").
#
# 2. __INIT__ (La Inauguración): Es el checklist que se ejecuta AUTOMÁTICAMENTE
#    cuando construyes el objeto. Aquí contratas gente y pones el capital inicial.
#
# 3. SELF (La Propiedad): Significa "MÍO".
#    - self.W = "MI matriz de pesos" (No la del vecino).
#    - Sin 'self', la empresa olvidaría sus datos al terminar la función.
#
# 4. DEF (Los Procesos): Son las SOPs (Procedimientos Operativos Estandar).
#    - Si no defines "backward", la oficina no sabrá cómo hacer auditoría.
# ==============================================================================

# --- PLANO 1: EL DEPARTAMENTO DE VENTAS (Capa Lineal) ---
class Linear():
    def __init__(self, input_size, output_size):
        '''
        LA INAUGURACIÓN (__init__)
        Aquí configuramos la oficina antes de abrir la puerta.
        input_size: Cuántos datos entran (Insumos).
        output_size: Cuántas decisiones salen (Resultados).
        '''
        # SELF.W -> Usamos 'self' para guardar a "NUESTROS" gerentes en memoria.
        # Inicializamos con valores aleatorios (Kaiming He) para no empezar en cero.
        self.W = np.random.randn(output_size, input_size) / np.sqrt(input_size/2)
        
        # SELF.b -> "NUESTRO" presupuesto base (Sesgo/Bias).
        self.b = np.zeros((output_size, 1))

    def __call__(self, X): 
        '''
        EL DÍA DE TRABAJO (Forward Pass)
        Aquí operamos: (Datos * Peso de la decisión) + Presupuesto Base
        Fórmula: y = wx + b
        '''
        self.x = X # Guardamos el input (Facturas de entrada) por si hay auditoría luego.
        Z = self.W @ X + self.b
        return Z

    def backward(self, Z_grad):
        '''
        LA AUDITORÍA (Backward Pass)
        Recibimos la queja (Z_grad) y buscamos culpables.
        '''
        # 1. Calculamos cuánto error vino de los insumos (para la capa anterior)
        X_grad = self.W.T @ Z_grad
        
        # 2. Calculamos qué tanto se equivocaron NUESTROS gerentes (self.W)
        self.W_grad = Z_grad @ self.x.T
        
        # 3. Calculamos qué tanto falló el presupuesto base (self.b)
        self.b_grad = np.sum(Z_grad, axis = 1, keepdims=True)
        
        return X_grad # Pasamos la bolita hacia atrás

    def update(self, learning_rate):
        '''
        AJUSTE DE SUELDOS (Update)
        Basado en la auditoría, corregimos a los gerentes.
        Nuevo Peso = Peso Actual - (Tasa de Aprendizaje * Culpa)
        '''
        self.W = self.W - learning_rate * self.W_grad
        self.b = self.b - learning_rate * self.b_grad

# --- PLANO 2: EL CONTROL DE CALIDAD (Capa ReLU) ---
class ReLU():
    def __call__(self, Z):
        '''
        FILTRO DE CALIDAD
        Si el número es negativo (pérdidas), lo volvemos 0.
        Si es positivo (ganancias), pasa directo.
        '''
        self.z = Z # Guardamos copia para auditoría
        return np.maximum(0, Z)

    def backward(self, A_grad):
        '''
        AUDITORÍA DEL FILTRO
        Solo aceptamos culpas si dejamos pasar el dato (si estaba activo).
        '''
        Z_grad = A_grad.copy()
        Z_grad[self.z <= 0] = 0  # Si estaba apagado, la culpa es 0.
        return Z_grad

    def update(self, learning_rate):
        # ReLU no tiene empleados (pesos), es solo una regla. No se actualiza.
        pass 

# --- PLANO 3: EL EDIFICIO CORPORATIVO (Sequential) ---
class Sequential_layers():
    def __init__(self, layers):
        '''
        EL ORGANIGRAMA
        layers: Es la lista de departamentos en orden.
        (Ej: [Ventas -> Filtro -> Marketing -> Salida])
        '''
        self.layers = layers

    def __call__(self, X):
        '''
        LA CADENA DE MANDO
        El dato entra y va pasando oficina por oficina.
        '''
        self.output = X
        for layer in self.layers:
            self.output = layer(self.output) # El output de uno es el input del siguiente
        return self.output

    def backward(self, loss_grad):
        '''
        REPARTICIÓN DE CULPAS GLOBAL
        Vamos desde la oficina del CEO (última capa) hacia la Recepción (primera).
        '''
        grad = loss_grad
        for layer in reversed(self.layers): # reversed = Ir hacia atrás
            grad = layer.backward(grad)
    
    def update(self, learning_rate):
        '''
        ORDEN DE MEJORA
        El CEO ordena a todos los departamentos que se actualicen.
        '''
        for layer in self.layers:
            layer.update(learning_rate)

# ==============================================================================
# SIMULACRO DE NEGOCIO (Entrenamiento)
# ==============================================================================

def softmax(x):
    # Convierte números feos en Probabilidades (0% a 100%)
    return np.exp(x) / np.sum(np.exp(x), axis=0, keepdims=True)

def train_simulacro():
    print("--- 🏗️ CONSTRUYENDO LA EMPRESA (MODELO) ---")
    
    # 1. Definimos la Estructura usando los PLANOS (Clases)
    model = Sequential_layers([
        Linear(10, 20), # Oficina 1: Entran 10 datos -> Salen 20 análisis
        ReLU(),         # Filtro: Elimina negativos
        Linear(20, 3),  # Oficina Final: Resume los 20 análisis en 3 decisiones finales
    ])
    
    # 2. Generamos DATOS DE PRUEBA (Clientes Falsos)
    print("--- 🎲 GENERANDO CLIENTES FALSOS ---")
    X_train = np.random.randn(10, 100) # 100 Clientes con 10 características c/u
    
    # Generamos las respuestas correctas (Etiquetas reales)
    Y_train = np.zeros((3, 100))
    indices = np.random.randint(0, 3, 100)
    Y_train[indices, np.arange(100)] = 1
    
    # 3. CICLO DE APRENDIZAJE
    print("--- 🚀 INICIANDO OPERACIONES (Training) ---")
    epochs = 100       # 100 Trimestres de operación
    learning_rate = 0.01 # Qué tan drásticos somos cambiando al personal
    
    for epoch in range(epochs):
        # A. OPERACIÓN (Forward)
        predictions_raw = model(X_train)
        probs = softmax(predictions_raw)
        
        # B. AUDITORÍA (Loss & Gradient)
        # Calculamos el error global y la "Primera Queja"
        loss_grad = (probs - Y_train) / 100 
        
        # Medimos qué tan mal estamos (Solo para ver en pantalla)
        cost = -np.sum(Y_train * np.log(probs + 1e-8)) / 100
        
        # C. REPARTIR CULPAS (Backward)
        model.backward(loss_grad)
        
        # D. MEJORAR PROCESOS (Update)
        model.update(learning_rate)
        
        if epoch % 10 == 0:
            print(f"Trimestre {epoch}: Costo (Error) = {cost:.4f}")

    print("--- ✅ FIN DEL EJERCICIO: LA EMPRESA HA APRENDIDO ---")

# Bloque de ejecución principal
if __name__ == "__main__":
    train_simulacro()

--- 🏗️ CONSTRUYENDO LA EMPRESA (MODELO) ---
--- 🎲 GENERANDO CLIENTES FALSOS ---
--- 🚀 INICIANDO OPERACIONES (Training) ---
Trimestre 0: Costo (Error) = 1.5108
Trimestre 10: Costo (Error) = 1.4279
Trimestre 20: Costo (Error) = 1.3612
Trimestre 30: Costo (Error) = 1.3068
Trimestre 40: Costo (Error) = 1.2618
Trimestre 50: Costo (Error) = 1.2247
Trimestre 60: Costo (Error) = 1.1940
Trimestre 70: Costo (Error) = 1.1686
Trimestre 80: Costo (Error) = 1.1473
Trimestre 90: Costo (Error) = 1.1292
--- ✅ FIN DEL EJERCICIO: LA EMPRESA HA APRENDIDO ---
